In [1]:
orders = [
  {"order_id":1003,"customer":"Ana","items":[{"product_id":301,"qty":2}]},
  {"order_id":1004,"customer":"Ben","items":[{"product_id":302,"qty":1},{"product_id":303,"qty":4}]}
]
 
order_items = []
for o in orders:
    for it in o["items"]:
        order_items.append({
            "order_id": o["order_id"],
            "customer": o["customer"],
            "product_id": it["product_id"],
            "qty": it["qty"]
        })
 
print(order_items)

[{'order_id': 1003, 'customer': 'Ana', 'product_id': 301, 'qty': 2}, {'order_id': 1004, 'customer': 'Ben', 'product_id': 302, 'qty': 1}, {'order_id': 1004, 'customer': 'Ben', 'product_id': 303, 'qty': 4}]


In [13]:
#Lab 7: Design an OLTP Normalized Schema (Orders)

In [7]:
import sqlite3
import pandas as pd

# Create (or connect to) the database
conn = sqlite3.connect("online_store.db")

# Create a cursor
cursor = conn.cursor()

# Drop existing tables (if they exist)

# Create tables
query = """

DROP TABLE IF EXISTS OrderItem;
DROP TABLE IF EXISTS "Order";
DROP TABLE IF EXISTS Product;
DROP TABLE IF EXISTS Customer;

CREATE TABLE Customer (
  customer_id SERIAL PRIMARY KEY,
  name VARCHAR(200),
  email VARCHAR(200) UNIQUE,
  default_shipping_address TEXT
);

CREATE TABLE Product (
    product_id INTEGER PRIMARY KEY AUTOINCREMENT,
    product_name TEXT,
    price REAL
);

CREATE TABLE "Order" (
    order_id INTEGER PRIMARY KEY AUTOINCREMENT,
    customer_id INTEGER,
    order_date DATE,
    shipping_address TEXT,
    FOREIGN KEY (customer_id) REFERENCES Customer(customer_id)
);

CREATE TABLE OrderItem (
    order_item_id INTEGER PRIMARY KEY AUTOINCREMENT,
    order_id INTEGER,
    product_id INTEGER,
    qty INTEGER,
    line_price REAL,
    FOREIGN KEY (order_id) REFERENCES "Order"(order_id),
    FOREIGN KEY (product_id) REFERENCES Product(product_id)
);

"""

cursor.executescript(query)

# Save changes
#conn.commit()

print("Tables created successfully!")

# ==========================================
# Insert data
# ==========================================

query = """

INSERT INTO Customer(name,email,default_shipping_address)
VALUES
('Alice','alice@gmail.com','Berlin'),
('Bob','bob@gmail.com','Munich'),
('Charlie','charlie@gmail.com','Hamburg');

INSERT INTO Product(product_name,price)
VALUES
('Laptop',1200),
('Mouse',25),
('Keyboard',60);

INSERT INTO "Order"(customer_id,order_date,shipping_address)
VALUES
(1,'2026-07-01','Berlin'),
(2,'2026-07-02','Munich'),
(1,'2026-07-05','Berlin');

INSERT INTO OrderItem(order_id,product_id,qty,line_price)
VALUES
(1,1,1,1200),
(1,2,2,50),
(2,3,1,60),
(3,2,1,25);

"""

cursor.executescript(query)

conn.commit()

print("Database created successfully!")

# ==========================================
# Display all tables
# ==========================================

query = """
SELECT name
FROM sqlite_master
WHERE type='table';
"""

tables = pd.read_sql_query(query, conn)

print("Available Tables")
display(tables)


# ==========================================
# Customer table
# ==========================================

query = """
SELECT *
FROM Customer;
"""

customer = pd.read_sql_query(query, conn)

print("Customer Table")
display(customer)

# ==========================================
# Product table
# ==========================================

query = """
SELECT *
FROM Product;
"""

product = pd.read_sql_query(query, conn)

print("Product Table")
display(product)

# ==========================================
# Order table
# ==========================================

query = """
SELECT *
FROM "Order";
"""

orders = pd.read_sql_query(query, conn)

print("Order Table")
display(orders)

# ==========================================
# OrderItem table
# ==========================================

query = """
SELECT *
FROM OrderItem;
"""

order_items = pd.read_sql_query(query, conn)

print("OrderItem Table")
display(order_items)

# ==========================================
# Close database connection
# ==========================================

conn.close()

Tables created successfully!
Database created successfully!
Available Tables


,name
0,sqlite_sequence
1,Customer
2,Product
3,Order
4,OrderItem


Customer Table


,customer_id,name,email,default_shipping_address
0,None,Alice,alice@gmail.com,Berlin
1,None,Bob,bob@gmail.com,Munich
2,None,Charlie,charlie@gmail.com,Hamburg


Product Table


,product_id,product_name,price
0,1,Laptop,1200.0
1,2,Mouse,25.0
2,3,Keyboard,60.0


Order Table


,order_id,customer_id,order_date,shipping_address
0,1,1,2026-07-01,Berlin
1,2,2,2026-07-02,Munich
2,3,1,2026-07-05,Berlin


OrderItem Table


,order_item_id,order_id,product_id,qty,line_price
0,1,1,1,1,1200.0
1,2,1,2,2,50.0
2,3,2,3,1,60.0
3,4,3,2,1,25.0


In [8]:
#Lab 8: Create a Denormalized Reporting Table

#Problem Statement
#Create a denormalized daily orders reporting table that contains customer_name, 
#order_id, order_date, total_order_value, product_list (as a single string), and shipping_city.


In [12]:
# Lab 8 Create a Denormalized Reporting Table



# ==========================================
# Connect to SQLite database
# ==========================================
conn = sqlite3.connect("online_store.db")
cursor = conn.cursor()

# ==========================================
# Create DailyOrderReport table
# ==========================================
query = """

DROP TABLE IF EXISTS DailyOrderReport;

CREATE TABLE DailyOrderReport (
    report_id SERIAL PRIMARY KEY,
  order_id INT,
  order_date DATE,
  customer_name VARCHAR(200),
  shipping_city VARCHAR(100),
  total_order_value NUMERIC(12,2),
  product_list TEXT
);

"""

cursor.executescript(query)

conn.commit()

print("DailyOrderReport table created successfully!")

# ==========================================
# Display table structure
# ==========================================

query = """
PRAGMA table_info(DailyOrderReport);
"""

cursor.execute(query)

print("\nTable Structure:")
for row in cursor.fetchall():
    print(row)

# ==========================================
# Close connection
# ==========================================

conn.close()

DailyOrderReport table created successfully!

Table Structure:
(0, 'report_id', 'SERIAL', 0, None, 1)
(1, 'order_id', 'INT', 0, None, 0)
(2, 'order_date', 'DATE', 0, None, 0)
(3, 'customer_name', 'VARCHAR(200)', 0, None, 0)
(4, 'shipping_city', 'VARCHAR(100)', 0, None, 0)
(5, 'total_order_value', 'NUMERIC(12,2)', 0, None, 0)
(6, 'product_list', 'TEXT', 0, None, 0)


In [15]:
#### Lab 10: Map Normalized Data to Star Schema (ETL sketch)
### Sketch SQL/ pseudocode to load fact table from normalized OLTP tables: orders, order_items, products, 
## customers. Include joins and aggregations for daily fact (one row per order_item with sales_amount).


# # ==========================================
# Import Library
# ==========================================

import sqlite3
import pandas as pd


# Connect to Database
conn = sqlite3.connect("lab10_dw.db")

print("Database connected successfully.")

# ==========================================
# Create Source Tables
# ==========================================

cursor = conn.cursor()

cursor.executescript("""

DROP TABLE IF EXISTS customers;
DROP TABLE IF EXISTS products;
DROP TABLE IF EXISTS orders;
DROP TABLE IF EXISTS order_items;

CREATE TABLE customers
(
    customer_id INTEGER PRIMARY KEY,
    customer_name TEXT,
    city TEXT
);

CREATE TABLE products
(
    product_id INTEGER PRIMARY KEY,
    product_name TEXT,
    product_price REAL
);

CREATE TABLE orders
(
    order_id INTEGER PRIMARY KEY,
    customer_id INTEGER,
    order_date DATE
);

CREATE TABLE order_items
(
    order_item_id INTEGER PRIMARY KEY,
    order_id INTEGER,
    product_id INTEGER,
    quantity INTEGER
);

""")

conn.commit()

print("Source tables created.")

Database connected successfully.
Source tables created.


In [16]:
# ==========================================
# Insert Sample Data
# ==========================================

cursor.executescript("""

INSERT INTO customers VALUES
(501,'Alice','Berlin'),
(502,'Bob','Munich');

INSERT INTO products VALUES
(201,'Laptop',1000),
(202,'Mouse',40),
(203,'Keyboard',80);

INSERT INTO orders VALUES
(1001,501,'2026-06-01'),
(1002,502,'2026-06-02');

INSERT INTO order_items VALUES
(1,1001,201,1),
(2,1001,202,2),
(3,1002,203,3);

""")

conn.commit()

print("Sample data inserted.")

Sample data inserted.


In [17]:
print("Customers")
display(pd.read_sql_query("SELECT * FROM customers;", conn))

print("Products")
display(pd.read_sql_query("SELECT * FROM products;", conn))

print("Orders")
display(pd.read_sql_query("SELECT * FROM orders;", conn))

print("Order Items")
display(pd.read_sql_query("SELECT * FROM order_items;", conn))

Customers


,customer_id,customer_name,city
0,501,Alice,Berlin
1,502,Bob,Munich


Products


,product_id,product_name,product_price
0,201,Laptop,1000.0
1,202,Mouse,40.0
2,203,Keyboard,80.0


Orders


,order_id,customer_id,order_date
0,1001,501,2026-06-01
1,1002,502,2026-06-02


Order Items


,order_item_id,order_id,product_id,quantity
0,1,1001,201,1
1,2,1001,202,2
2,3,1002,203,3


In [18]:
# Create Fact Table

# ==========================================
# Create FactOrders
# ==========================================

cursor.execute("DROP TABLE IF EXISTS FactOrders")

cursor.execute("""

CREATE TABLE FactOrders
(
    order_id INTEGER,
    customer_key INTEGER,
    product_key INTEGER,
    order_date DATE,
    quantity INTEGER,
    unit_price REAL,
    sales_amount REAL
)

""")

conn.commit()

print("FactOrders created.")

FactOrders created.


In [19]:
# ==========================================
# ETL Load FactOrders
# ==========================================

cursor.execute("""

INSERT INTO FactOrders

SELECT

o.order_id,
c.customer_id,
p.product_id,
o.order_date,
oi.quantity,
p.product_price,
oi.quantity * p.product_price AS sales_amount

FROM orders o

JOIN customers c
ON o.customer_id = c.customer_id

JOIN order_items oi
ON o.order_id = oi.order_id

JOIN products p
ON oi.product_id = p.product_id;

""")

conn.commit()

print("Fact table loaded.")

Fact table loaded.


In [20]:
fact = pd.read_sql_query(
    "SELECT * FROM FactOrders;",
    conn
)

fact

,order_id,customer_key,product_key,order_date,quantity,unit_price,sales_amount
0,1001,501,201,2026-06-01,1,1000.0,1000.0
1,1001,501,202,2026-06-01,2,40.0,80.0
2,1002,502,203,2026-06-02,3,80.0,240.0


In [ ]:
#Lab 11
# Create Customer Dimension (Type 2)